In [6]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.5)

In [7]:
hard    = pd.read_csv('figures/pareto_hard_results.csv')
exploit = pd.read_csv('figures/pareto_exploit_results.csv')
rag     = pd.read_csv('figures/pareto_rag_results.csv')

print(f"hard protocols:    {hard['protocol'].unique().tolist()}")
print(f"exploit protocols: {exploit['protocol'].unique().tolist()}")
print(f"rag protocols:     {rag['protocol'].unique().tolist()}")

hard protocols:    ['blue_defer', 'blue_edit', 'resample', 'trusted_monitoring', 'upfront_auditing', 'trusted_policy']
exploit protocols: ['blue_defer', 'blue_edit', 'resample', 'trusted_monitoring', 'upfront_auditing', 'trusted_policy']
rag protocols:     ['blue_defer', 'blue_edit', 'resample', 'trusted_monitoring', 'upfront_auditing', 'trusted_policy']


In [ ]:
seaborn_palette = sns.color_palette()
blue_edit_color  = seaborn_palette[0]
blue_defer_color = seaborn_palette[1]
resample_color   = seaborn_palette[2]

protocols_colors = [
    ('blue_edit',  blue_edit_color),
    ('blue_defer', blue_defer_color),
    ('resample',   resample_color),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Panel (a): Severity exploit ────────────────────────────────────────────────
ax = axes[0]
for protocol, color in protocols_colors:
    for metric_key, ls, alpha in [('blue_0', 'dotted', 0.15), ('blue_eh', 'solid', 0.2)]:
        d = exploit[(exploit['protocol'] == protocol) & (exploit['metric'] == metric_key)].dropna(subset=['threshold'])
        if len(d) > 0:
            ax.scatter(d['safety'], d['usefulness'], s=30, alpha=alpha, color=color, marker='o')
            sns.regplot(data=d, x='safety', y='usefulness', lowess=True, scatter=False, ax=ax,
                        color=color, line_kws={'linestyle': ls, 'linewidth': 2})
ax.set_xlabel('Safety')
ax.set_ylabel('Usefulness')
ax.set_xlim(0, 1)
ax.set_title('(a) Severity exploit')
ax.grid(True, linestyle='--')

# ── Panel (b): Drug information access ────────────────────────────────────────
ax = axes[1]
for protocol, color in protocols_colors:
    d_base = rag[(rag['protocol'] == protocol) & (rag['source'] == 'baseline')]
    if len(d_base) > 0:
        ax.scatter(d_base['safety'], d_base['usefulness'], s=30, alpha=0.15, color=color, marker='o')
        sns.regplot(data=d_base, x='safety', y='usefulness', lowess=True, scatter=False, ax=ax,
                    color=color, line_kws={'linestyle': 'dotted', 'linewidth': 2})
    d = rag[(rag['protocol'] == protocol) & (rag['source'] == 'rag')].dropna(subset=['threshold'])
    if len(d) > 0:
        ax.scatter(d['safety'], d['usefulness'], s=30, alpha=0.2, color=color, marker='o')
        sns.regplot(data=d, x='safety', y='usefulness', lowess=True, scatter=False, ax=ax,
                    color=color, line_kws={'linestyle': 'solid', 'linewidth': 2})
ax.set_xlabel('Safety')
ax.set_ylabel('Usefulness')
ax.set_xlim(0, 1)
ax.set_title('(b) Drug information access')
ax.grid(True, linestyle='--')

# ── Legends ────────────────────────────────────────────────────────────────────
color_entries = [
    Line2D([0], [0], color=blue_edit_color,  lw=2, linestyle='solid', label='trusted_editing'),
    Line2D([0], [0], color=blue_defer_color, lw=2, linestyle='solid', label='defer_to_trusted'),
    Line2D([0], [0], color=resample_color,   lw=2, linestyle='solid', label='resample'),
]

axes[0].legend(handles=color_entries + [
    Line2D([0], [0], color='gray', lw=2, linestyle='dotted', label='probability'),
    Line2D([0], [0], color='gray', lw=2, linestyle='solid',  label='probability + severity'),
], fontsize='small', loc='best')

axes[1].legend(handles=color_entries + [
    Line2D([0], [0], color='gray', lw=2, linestyle='dotted', label='Without drug info'),
    Line2D([0], [0], color='gray', lw=2, linestyle='solid',  label='With drug info'),
], fontsize='small', loc='best')

plt.tight_layout()
plt.savefig('figures/comparison_addons.png', bbox_inches='tight')
plt.show()